Bitcoin, per come è progettato, ha un problema di **scalabilità**: la dimensione dei blocchi e il tempo medio di creazione di un blocco limitano fortemente il numero di transazioni che possono essere registrate direttamente sulla blockchain.

Abbiamo visto con SegWit che un blocco può arrivare a circa 4 MB di dimensione (anche se in media è più vicino a 2 MB) e in media si riescono a registrare al più alcune migliaia di transazioni per blocco. Dal momento che un blocco viene prodotto mediamente ogni 10 minuti, il numero di transazioni al secondo rimane molto basso rispetto ai sistemi di pagamento centralizzati come Visa o Mastercard, che ne possono gestire migliaia al secondo. **Questo rappresenta un limite molto importante se l'obiettivo finale è usare Bitcoin non solo come riserva di valore, ma come vero e proprio sistema di pagamento globale.**

Per questa ragione si distingue tra Bitcoin come **layer 1 (base layer)**, che è la blockchain principale, e soluzioni di **layer 2**, che sono **protocolli costruiti sopra il layer 1 e che non registrano ogni singolo pagamento direttamente on-chain, permettendo così di risolvere in gran parte il problema di scalabilità.**

La **Lightning Network** rappresenta una soluzione di layer 2 che permette di effettuare molti pagamenti off-chain, usando la blockchain solo in alcuni momenti chiave, come l'apertura o la chiusura di un canale oppure quando una delle due parti si comporta in modo scorretto.

**Intuitivamente possiamo vedere molto ad alto livello la Lightning Network come un'analogia con un contratto**: due parti (Alice e Bob) decidono di aprire un canale di pagamento tra di loro depositando una certa quantità di Bitcoin. Finché entrambi rispettano le regole del contratto, possono aggiornare privatamente il loro bilancio senza coinvolgere la blockchain. Quest'ultima entra in gioco solo se serve da "arbitro": nel caso in cui una delle due parti non rispetti il contratto(provando come vedremo a pubblicare uno stato vecchio del canale, cioè una situazione non più valida ma per lei vantaggiosa) --> il protocollo è costruito in modo da poter punire la parte che cerca di imbrogliare.

L'idea di Lightning Network era già in lavorazione dal 2014, ma è stata implementata solo con SegWit nel 2017 per via del fatto che SegWit ha risolto un problema di malleabilità delle transazioni che era un ostacolo importante per la realizzazione di questo protocollo (vedi SegWit per maggiori dettagli).

## Lightning Network: funzionamento
Con la Lightning Network non si vuole costruire soltanto un singolo canale tra due utenti, ma una vera e propria rete di canali di pagamento off-chain. In questa sezione vedremo solo il caso più semplice bidirezionale tra due utenti (Alice e Bob) per capire bene il funzionamento, ma nella prossima lezione si approfondirà come è possibile costruire una rete di canali interconnessi.

es. Alice non ha un canale diretto con Carol, ma ha un canale con Bob e Bob ha un canale con Carol. Alice può sfruttare la Lightning Network per pagare Carol passando per Bob, evitando di dover aprire un canale diretto con Carol e anche di dover registrare ogni singolo pagamento sulla blockchain. Vedremo tutto questo nella prossima lezione.

### Apertura del canale: **Funding Transaction**
Immaginiamo che Alice voglia aprire un canale di pagamento con Bob e voglia mettere sul canale 10 bitcoin. 

Questi 10 bitcoin rappresentano la **capacità del canale**: è la quantità di fondi bloccati nel canale. Questa informazione, per il modello semplificato che andremo a studiare, è visibile a tutti in quanto deriva dalla transazione di apertura del canale (la cosiddetta **funding transaction**).

I successivi aggiornamenti del **bilancio** tra Alice e Bob invece avvengono tramite messaggi privati, e quindi non vengono immediatamente pubblicati nella blockchain --> **non sono visibili a tutti**. Solo quando il canale viene chiuso, viene pubblicata una transazione che riflette l'ultimo stato del canale, cioè l'ultimo bilancio concordato tra Alice e Bob.

Lo **stato iniziale del canale** è quindi:
```text
Alice 10  --------  0 Bob
```
Se poi Alice decide di pagare 2 bitcoin a Bob, il **bilancio del nuovo stato del canale** diventa:
```text
Alice 8  --------  2 Bob
```
Lo scopo del protocollo è fare in modo che **ogni aggiornamento del bilancio sia sicuro: nessuna delle due parti deve poter pubblicare uno stato vecchio e per lui vantaggioso senza rischiare una penalizzazione.**

Per aprire il canale, Alice costruisce una **Funding Transaction**. Questa transazione prende in input alcuni bitcoin che Alice possiede già sulla blockchain e **come output un multisig 2-of-2 spendibile solo con la firma sia di Alice che di Bob**.  Uno script semplificato di questo tipo (senza pensare alla sicurezza relativa a ECDSA e quindi mostrando pubblicamente le pk) è:
```text
scriptPubKey (locking script):
2 <pkA> <pkB> 2 OP_CHECKMULTISIG

scriptSig (unlocking script): 
0 <sigA> <sigB>
```
dove si ricorda che OP_CHECKMULTISIG prende un intero $n$ dallo stack, legge $n$ chiavi pubbliche, poi prende un altro inter $m$ dallo stack e legge $m$ firme, e restituisce true se almeno $m$ delle $n$ chiavi pubbliche corrispondono a $m$ delle firme (e si ricorda che lo zero nello scriptSig è un workaround per un bug di OP_CHECKMULTISIG).

#### Due Problemi
Se Alice pubblicasse subito la FundingTx sulla blockchain, i suoi 10 bc finirebbero in un output spendibile solo con la collaborazione di Bob. Ma **se Bob sparisse, si rifiutasse di collaborare o perdesse la chiave, allora Alice non potrebbe più spendere quell'output perché mancherebbe la firma di Bob, perdendo per sempre i suoi 10 bitcoin**.

A primo impatto si potrebbe pensare che per risolvere il problema basterebbe che Bob inviasse ad Alice la firma della FundingTx prima che questa venga pubblicata da lei sulla blockchain: del resto Bob non sembra avere alcun incentivo a non farlo visto che i soldi non li sta mettendo lui. Però in realtà questo comporta un **secondo problema**.

Supponiamo infatti che il canale venga aperto con bilancio iniziale 10-0 a favore di Alice, e che Alice paghi a Bob 2 bitcoin, aggiornando il bilancio a 8-2. Bob potrebbe nel frattembo inviare il bene costato 2 bitcoin ad Alice, ma nel frattempo Alice potrebbe decidere di pubblicare sulla blockchain la funding transaction riguardagnando i suoi 10 bitcoin e nel frattempo avendo fregato pure il bene a Bob! **Serve quindi un meccanismo che permetta di rendere le transazioni revocabili dal punto di vista economico**, ma dal momento che in bitcoin una transazione valida una volta pubblicata non può mai essere annullata, l'idea di revocabilità è implementata con il concetto di punizione --> **la revoca starà nel fatto che se Alice prova a pubblicare la funding transaction dopo aver aggiornato il bilancio a 8-2, Bob avrà il tempo di "punirla" prendendosi lui tutti i fondi**. Questa è l'idea alla base della **Refunding Transaction** e poi delle successive **Commitment Transactions**.

### **OP_CHECKSEQUENCEVERIFY e Funding Transaction**
Per capire RefundingTx, CommitmentTx e revoca, si introduce anzitutto l'opcode bitcoin **OP_CHECKSEQUENCEVERIFY (CSV)**. Questo opcode permette di imporre un vincolo temporale sulla spesa di una vecchia transazione: **un certo output può essere speso solo dopo che è passato un certo numero di blocchi da quello in cui la transazione è stata confermata**. 

Se abbiamo uno scriptPubKey del tipo:
```text
10 OP_CHECKSEQUENCEVERIFY OP_DROP TRUE
```
significa che l'output a cui punta l'input con questo scriptPubKey può essere speso solo dopo che sono passati almeno 10 blocchi da quello in cui è stata confermata la transazione che lo contiene. In particolare anzitutto OP_CHECKSEQUENCEVERIFY controlla che la differenza tra il blocco attuale e il blocco in cui è stata confermata la transazione sia almeno 10, se vero allora non fa nulla e si passa a OP_DROP che rimuove il 10 dallo stack, e poi True che quindi valida la transazione. Se invece la differenza è minore di 10, allora CSV restituisce false e quindi la transazione non è valida.

Ora, se volessimo che un output sia spendibile solo da Alice ma dopo ad esempio 100 blocchi possiamo tranquillamente aggiungere il controllo della sua firma:
```text
scriptPubKey:
100 OP_CHECKSEQUENCEVERIFY OP_DROP <pkA> OP_CHECKSIG

scriptSig:
<sigA>
```

A partire da questo meccanismo si definisce l'idea generale di **revoca**: **Alice può recuperare i suoi 10 bitcoin se Bob non collabora, ma solo dopo un certo numero di blocchi. Bob invece può prendere tutti i 10 bitcoin subito se possiede una certa chiave di revoca che Alice gli ha rivelato.**

In particolare se Alice pubblicasse una vecchia transazione che le assegna più soldi di quanto dovrebbe avere nel bilancio del canale aggiornato, allora Bob può usare la chiave di revoca ricevuta per sottrarle i fondi prima che Alice possa spenderli (perché devono passare un certo numero di blocchi).

(ricorda che per mettere una transazione da un output multisig 2-of-2 serve la firma $\underline{\text{della nuova transazione}}$ di entrambi)

Per attuare l'idea si segue il seguente schema:
1. Alice **prepara la FundingTx**:
    ```text
    FundingTx, txid: 123...ff:
    input: fondi precedenti di Alice nella blockchain
    output: 10 bitcoin spendibili da pkA e pkB (multisig 2-of-2)
    ```
    Alice però aspetta prima di pubblicare la FundingTx sulla blockchain. 
2. Alice crea una nuova coppia di chiavi **(skAr0, pkAr0)**, che rappresentano la chiave di revoca.
3. Alice **prepara anche la RefundingTx**, che spende l'output della FundingTx ed è costruita come segue:
    ```text
    RefundingTx, txid: 33...ef:
    input: output della FundingTx 123...ff
    output: 10 bitcoin spendibili da:
        - pkA dopo 144 blocchi (circa 24 ore)
        oppure
        - pkB & pkAr0 (subito, multisig 2-of-2)
    ```
    chiariremo a breve il senso di pkAr0. Per ora è sufficiente capire che rappresenta la chiave pubblica di revoca generata da Alice, e che quindi se Bob conosce la firma della Refunding Tx fatta con skAr0, allora può spendere subito i 10 bitcoin senza dover aspettare 144 blocchi.
4. **Bob firma la RefundingTx e invia la firma ad Alice.** Alice aspetta prima di pubblicare la funding transaction sulla blockchain prima di ricevere questa firma: una volta ricevuta infatti lei può essere sicura di potersi riprendere i suoi 10 bitcoin anche se Bob non collabora pubblicando la RefundingTx firmata da entrambi, riottenendo i suoi soldi dopo 144 blocchi. **Alice quindi pubblica la FundingTx sulla blockchain non appena riceve òa firma di Bob sulla RefundingTx, aprendo così il canale.**

A questo punto il canale è aperto, e il bilancio iniziale è 10-0 a favore di Alice. Ma Alice non ha ancora dato a Bob la chiave segreta di revoca skAr0, in questo modo se Bob sparisse Alice potrebbe pubblicare la RefundingTx riprendendosi i suoi 10 bitcoin dopo 144 blocchi, senza rischiare che Bob possa rubarglieli subito dal momento che lui non conosce la skAr0. 

**Solo quando il canale viene aggiornato per la prima volta, ad esempio se Alice manda 2 bitcoin a Bob, allora Alice rivela a Bob skAr0**. A quel punto la **ormai vecchia** RefundingTx che assegna 10 bitcoin ad Alice è "revocata" perché diventa pericolosa per Alice se decide di fare la furba e pubblicarla: **se Alice la pubblicasse Bob potrebbe tranquillamente usare la chiave di revoca skAr0 per firmarla e prendersi i suoi 10 bitcoin**.

**Ma Bob, una volta ricevuta skAr0, non può pubblicare la RefundingTx e rubare i 10 bitcoin ad Alice indipendentemente dal fatto che sia lei a pubblicarla?** NO, perché solo Alice può pubblicare la RefundingTx dal momento che solo lei ha la firma della RefundingTx fatta da entrambi (Bob gliel'ha inviata prima), mentre Bob non ha la firma della RefundingTx fatta da Alice, e quindi dal momento che la Funding richiede checkmultisig 2-of-2 anche se Bob pubblicasse la RefundingTx senza la firma di Alice questa non sarebbe valida. 

Per costruire uno script che permetta di implementare la RefundingTx è necessario lavorare con gli opcode OP_IF e OP_ELSE, che permettono di costruire script con due rami alternativi. 

In pratica funzionano come segue: se nello stack trovo TRUE --> eseguo il ramo OP_IF, se invece trovo FALSE --> eseguo il ramo OP_ELSE. 

Vogliamo uno script che permetta di costruire la seguente logica:
```text
se ramo revoca:
    Bob + chiave di revoca possono spendere subito
altrimenti:
    Alice può spendere dopo 144 blocchi
```

Possiamo implementarlo così:
```text
scriptPubKey:
OP_IF
    2 <pkAr0> <pkB> 2 OP_CHECKMULTISIG
OP_ELSE
    144 OP_CHECKSEQUENCEVERIFY OP_DROP <pkA> OP_CHECKSIG
OP_ENDIF

scriptSig Bob:
0 <sigAr0> <sigB> TRUE

scriptSig Aslice:
<sigA> FALSE
```

Lo script funziona perché se è Bob a voler spendere, allora mette il suo scriptSig 0 <sigAr0> <sigB> TRUE --> nello stack finisce in cima TRUE. Quando arriva lo ScriptPubKey, entra anzitutto OP_IF che vede TRUE e quindi decide di eseguire solo il suo ramo. Al contrario se fosse stata Alice a voler spendere, allora OP_IF avrebbe visto FALSE e quindi avrebbe fatto inserire nello stack il secondo ramo, che prevede un vincolo temporale di 144 blocchi e poi la firma di Alice.


### **Commitment Transaction**
Siamo arrivati al punto in cui il nuovo bilancio del canale è 8-2 a favore di Alice, e Alice ha rivelato a Bob la chiave di revoca skAr0. **Per rappresentare questo nuovo stato del canale, vengono create due nuove transazioni chiamate Commitment Transaction, una per Alice e una per Bob: CommitTx1(Alice) e CommitTx1(Bob)**.

Entrambe le transazioni spendono lo stesso output della FundingTx (output multisig 2-of-2 da 10 bitcoin) --> la CommitTx1 di Alice deve essere firmata da Bob per essere valida, mentre la CommitTx1 di Bob deve essere firmata da Alice per essere valida.

- **Commitment Transaction di Alice**: è la transazione che **tiene in mano Alice e che può pubblicare se vuole chiudere unilateralmente il canale** perché Bob sparisce. Ha la seguente struttura:
    ```text
    CommitTx1(Alice)
    input: output della FundingTx 123...ff
    output: 
        - 8 bitcoin spendibili da pka (Alice) dopo 144 blocchi oppure da pkb e pkAr1 (ossia da Bob subito, ma solo se conosce la chiave di revoca skAr1, che Alice gli deve aver rivelato) 
        - 2 bitcoin spendibili da pkB (Bob) subito
    ```
    il senso della CommitTx1(Alice) è questo: se Bob sparisce dopo che è stato fatto l'aggiornamento del bilancio a 8-2 (e quindi Alice gli ha rivelato la chiave di revoca skAr0), allora Alice può pubblicare questa sua CommitTx1 di modo che Bob riceva effettivamente i 2 bitcoin che gli spettano, mentre Alice riceve la sua parte di 8 bitcoin dopo 144 blocchi.  
    **Perché Alice deve aspettare?** Per lo stesso identico discorso di prima: se invece Bob non era scomparso e il canale era stato di nuovo aggiornato, magari con Alice che gli aveva mandato altri 2 bitcoin, allora Alice avrebbe potuto pubblicare questa vecchia CommitTx1 che le assegna 8 bitcoin invece di 6, ma Bob ha ricevuto la chiave segreta skAr1 prima che fosse confermato l'aggiornamento del bilancio a 6-4, e quindi può penalizzare Alice prendendosi lui gli 8 bitcoin subito se lei decide di fare una cosa del genere.  

    Lo script dell'output da 8 bitcoin sarebbe quindi del tutto simile a quello della RefundingTx, solo considerando la nuova chiave di revoca skAr1 invece di skAr0:
    ```text
    OP_IF
    2 <pkAr1> <pkB> 2 OP_CHECKMULTISIG
    OP_ELSE
        144 OP_CHECKSEQUENCEVERIFY OP_DROP <pkA> OP_CHECKSIG
    OP_ENDIF
    ```
    Ribadiamolo: se Alice pubblica onestamente questa CommitmentTx, aspetta 144 blocchi e poi prende i suoi 8 bitcoin.

    Se invece in futuro questa CommitmentTx diventa vecchia (bilancio aggiornato) e Alice prova comunque a pubblicarla, Bob, avendo ricevuto skAr1, può usare il ramo immediato e prendere gli 8 bitcoin di Alice.
- **Commitment Transaction di Bob**: stesso identico discorso, ma ragionando dal punto di vista di Bob. La CommitTx1(Bob) è la transazione che **Bob tiene in mano e che può pubblicare se vuole chiudere unilateralmente il canale** perché Alice sparisce.
    ```text
    CommitTx1(Bob)
    input: output della FundingTx 123...ff
    output: 
        - 8 bitcoin spendibili da pkA (Alice) subito
        - 2 bitcoin spendibili da pkB (Bob) dopo 144 blocchi oppure da pkA e pkBr1 (ossia da Alice subito, ma solo se conosce la chiave di revoca skBr1, che Bob gli deve aver rivelato) 
    ```
    In questo caso se Alice sparisce dopo che è stato fatto l'aggiornamento del bilancio a 8-2 allora Bob deve essere in grado di prendersi i suoi 2 bitcoin (se li prende dopo 144 blocchi) mentre ad Alice arrivano i suoi 8 bitcoin.  
    Il motivo per cui Bob deve aspettare è per dare il tempo ad Alice di punirlo se lui prova a fare il furbo: immaginiamo che Bob ad esempio invii 1 bitcoin ad Alice, aggiornando il bilancio a 9-1. Prima di aggiornare il bilancio, affinché Alice fosse d'accordo, Bob deve averle rivelato la chiave di revoca skBr1, e quindi se Bob prova a pubblicare questa vecchia CommitTx1(Bob) che gli assegna 2 bitcoin invece di 1, Alice può usare la chiave di revoca skBr1 per prendersi i 2 bitcoin di Bob subito, senza dover aspettare 144 blocchi.

Il motivo di due CommitmentTx è che **ciascuna delle due parti deve essere in grado di chiudere il canale da sola, senza aspettare che l'altra parte collabori**. In questo senso, come accennato, affinché ognuna di queste parti possa effettivamente pubblicare queste CommitTx, è necessario che **CommitTx1(Alice)** sia firmata da Bob e che la firma sia inviata ad Alice, e che **CommitTx1(Bob)** sia firmata da Alice e che la firma sia inviata a Bob. 

Chiaramente come già detto le chiavi di revoca vengono **rivelate rispettivamente solo nel momento in cui deve essere effettuato l'aggiornamento del bilancio, di modo che queste vecchie CommitTx diventino pericolose se pubblicate**. 

Vediamo ora un recap di tutto il processo per avere chiaro l'ordine degli avvenimenti:
1. **Alice prepara la FundingTx con 10 bitcoin output come multisig 2-of-2 con Bob, ma aspetta prima di pubblicarla**
2. **Alice prepara la RefundingTx che spende l'output della FundingTx**. Questa spende i 10 bitcoin a favore di Alice dopo 144 blocchi oppure a favore di Bob subito se lui conosce la chiave di revoca skAr0.
3. **Bob firma la RefundingTx e invia la firma ad Alice**. **Alice allora pubblica tranquillamente la FundingTx sulla blockchain** (se lui sparisse, lei potrebbe comunque pubblicare refunding e prendersi i suoi 10 bitcoin dopo 144 blocchi).
4. Si vuole aggiornare il bilancio a 8-2 a favore di Alice, quindi **Alice rivela a Bob la chiave di revoca skAr0**. Quindi Alice non può più pubblicare la RefundingTx perché Bob potrebbe usare skAr0 per prendersi i 10 bitcoin subito.
5. **Alice prepara la CommitTx1(Alice) e Bob prepara la CommitTx1(Bob)**. Rappresentano rispettivamente le transazioni che i due possono usare indipendentemente per chiudere unilateralmente il canale. Ognuna punta alla funding transaction, e assegna i bitcoin secondo l'ultimo bilancio aggiornato (8-2).
6. **Scambio delle firme: Bob firma la CommitTx1(Alice) e invia la firma ad Alice, mentre Alice firma la CommitTx1(Bob) e invia la firma a Bob**. Ora entrambi possono usare le CommitTx per chiudere unilateralmente il canale se l'altra parte sparisce. 
7. **Aggiornamento del bilancio**: Prima di aggiornare il bilancio, Alice deve rivelare a Bob la chiave di revoca skAr1, e Bob deve rivelare ad Alice la chiave di revoca skBr1. Ora le vecchie CommitTx1 sono "revocate" (se pubblicate danno la possibilità all'altra parte di punire chi le pubblica).  
Dopo aver aggiornato il bilancio, Alice e Bob creano due nuove CommitTx2 adatte e si ripete il processo, fino a chiusura del canale (che sia collaborativa o unilaterale).

Si parla di **Lightning Network** in questo senso perché i pagamenti off-chain quindi richiedono solo l'invio di questi pochi messaggi tra le due parti, velocissimo.

### Closing Transaction 
**Cosa avviene alla fine della vita del canale?** Supponendo che Alice e Bob siano stati collaborativi fino alla fine, allora se si vuole chiudere il canale basta che entrambi collaborino per pubblicare una transazione 2-of-2 multisig di chiusura del canale che spende l'output della FundingTx e che assegna i bitcoin a ciascuno secondo l'ultimo bilancio aggiornato. Questa transazione è detta **Closing Transaction**. Chiaramente se poi nel momento di mandare la firma uno dei due sparisce, allora l'altro può sempre chiudere unilateralmente il canale usando la sua CommitTx più recente, e aspettare 144 blocchi per prendere i suoi soldi.

Il vantaggio della chiusura collaborativa è che ovviamente non necessita che la parte onesta deve aspettare 144 blocchi per recuperare i suoi fondi. 

### "Limiti" di Lightning Network
La Lightning Network è una soluzione molto potente per risolvere il problema di scalabilità di Bitcoin, ma ha comunque alcuni limiti importanti da tenere in considerazione:
1. **Necessità di monitorare la blockchain**: è necessario che i partecipanti alla Lightning Network monitorino la blockchain con continuità. Infatti, se una delle due parti pubblica una vecchia Commitment Transaction, l'altra parte deve essere in grado di rilevare questa transazione e reagire entro un certo numero di blocchi per punire l'altra parte usando la revocation key corretta.  Quindi, se un nodo resta offline troppo a lungo, rischia di non vedere in tempo il tentativo di frode.  

    Per questo motivo i nodi Lightning (soprattutto quelli che comunicano con moltissimi altri nodi) devono restare online o ad affidarsi a servizi esterni di monitoraggio. 
2. **Memorizzare le revocation keys**: ogni volta che un canale viene aggiornato, lo stato precedente viene revocato rivelando alla controparte una chiave di revoca. Questo significa che se ci sono moltissimi aggiornamenti del canale bisogna conservare le informazioni necessarie per punire la pubblicazione di una qualsiasi delle commitment transaction revocate. 
   
   Per questo motivo in linea teorica il numero di chiavi da memorizzare può crescere molto. Nella pratica però non si memorizzano necessariamente tutte le chiavi di revoca, esistono meccanismi più efficienti di pruning e compressione per ridurre lo spazio necessario.
3. **Stati sbilanciati del canale**: è preferibile evitare che un canale Lightning torni con bilancio a 0 in una delle due direzioni. Questo perché se ad esempio si è partiti con 10-0 per Alice e si arriva a 0-10, allora Alice non ha motivo di non provare a fregare bob pubblicando una delle vecchie CommitTx, visto che tanto anche se Bob la punisse comunque lei gli doveva tutti i soldi. 
   
   Per questo motivo è preferibile che i canali siano sempre bilanciati, o comunque che non arrivino a 0 in una delle due direzioni, per evitare che una delle due parti abbia un incentivo a pubblicare una vecchia transazione.
4. **Problema fee dinamiche**: in tutto il ragionamento non abbiamo considerato che le transazioni devono tener conto anche di fee verso i miner, quindi ogni volta che si aggiorna il bilancio ad es. da 10-0 a 8-2 in realtà si devono prendere in considerazione anche le fee, e quindi il bilancio reale potrebbe essere ad esempio 7.999-1.999 invece di 8-2.

    Le transaction fee su Bitcoin sono molto dinamiche, quindi una CommitmentTx costruita oggi potrebbe avere fee adeguata attualmente, ma magari insufficiente domani se la mempool è congestionata. Il rischio è che quindi domani Alice provi a fregarmi pubblicando una vecchia CommitTx e io non riesca a pubblicare la penalty transaction in tempo per punirla perché ha fee troppo bassa --> passano 144 blocchi e la penalty non viene accettata, mi perdo i soldi (qui da chiedere al prof... ma la penalty transaction non la faccio al momento? quindi non potrei mettere fee adeguata?)

    Per risolvere il problema meccanismi tipo **Child pays for Parent (CPFP)** permettono di costruire una transazione figlia con fee più alta che spende l'output della CommitmentTx, incentivando così i miner a includere anche la CommitmentTx nella blockchain.

    (magari qui il problema piuttosto è se Alice sparisce e io devo chiudere unilateralmente il canale, ma txfee troppo bassa e quindi i miner non la mettono mai nella blockchain)

